# Phase 6: every frozen system on the 800 main answers

Runs after the pre-registration is filed. Stores Channel A's spans and fusion features (and its two ablations' detector records), the neural outcome model's probabilities, and every frozen repairer's output with the rule checker's and Channel A's spans: repairer v1, stage A, no-KEEP, stage B and the free-rewrite baseline (D-101). Two queues run side by side, one per T4. Attach: the faithguard-main-items dataset (no gold), the faithguard-detector-models dataset, and the outputs of the notebooks detector_ablation, repair_v1, repair_sft, repair_spans, repair_self_train, repair_rewrite_v1 and outcome_model; attach the faithguard-gold-spans dataset (span offsets cut from the labels by `faithguard labelling spans`) to run v1 and the rewrite baseline on the labelled spans too (R-H1). The outputs are scored on a laptop against the labels. Expect about 9 GPU hours.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import shutil, socket
try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: verify your phone number (Settings), then switch Internet on in the right-hand panel.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU in this session. Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
# Kaggle's image ships torchao 0.10, which PEFT 0.21 rejects when it adds LoRA to fp16 weights; nothing here uses it
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
# this kernel started before the install, and the clone's own folder would shadow the package: import from src/
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Inputs: the items, the detectors, the neural model and the repairers' adapters

In [ ]:
import glob, os, subprocess, sys
for flat in glob.glob('/kaggle/input/**/*__model__*', recursive=True):  # the models dataset, folders flattened
    target = os.path.join(WORK, 'prev', *os.path.basename(flat).split('__'))
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        os.symlink(flat, target)
def first(pattern):
    found = sorted(glob.glob(pattern, recursive=True))
    return found[0] if found else None
ITEMS = first('/kaggle/input/**/items.jsonl')
DETECTORS = {'channel-a': os.path.join(WORK, 'prev', 'channel-a', 'model'), 'channel-a-noslot': os.path.join(WORK, 'prev', 'channel-a-noslot', 'model'),
             'channel-a-noxbrl': first('/kaggle/input/**/channel-a-noxbrl/model')}
NEURAL = first('/kaggle/input/**/outcome-model/model')
ADAPTERS = {'repair-v1': first('/kaggle/input/**/repair-v1/adapter'), 'repair-sft': first('/kaggle/input/**/repair-sft/adapter'),
            'repair-sft-nokeep': first('/kaggle/input/**/repair-sft-nokeep/adapter'), 'stage-b': first('/kaggle/input/**/self-train/round-2/adapter'),
            'rewrite-v1': first('/kaggle/input/**/rewrite-v1/adapter')}
GOLD_SPANS = first('/kaggle/input/**/gold-spans.jsonl')  # R-H1's labelled spans, if attached
print(ITEMS, DETECTORS, NEURAL, ADAPTERS, GOLD_SPANS, sep='\n')
assert ITEMS and all(DETECTORS.values()) and NEURAL and all(ADAPTERS.values()), 'attach every input listed above'

In [ ]:
# every adapter and detector used here must match its frozen hash (D-101, D-104, D-110)
import hashlib, json
def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()
frozen = [f for m in ('frozen-repair', 'frozen-detector') for f in json.load(open(f'manifests/{m}.json'))['files']]
for name, folder in {**ADAPTERS, **DETECTORS}.items():
    key = '/' + '/'.join(folder.replace(os.sep, '/').rstrip('/').split('/')[-2:]) + '/'
    files = [f for f in frozen if key in f['path'].replace('__', '/')]  # the models dataset's folders are flattened with __
    assert files, f'{name}: no frozen hashes for {key}'
    for f in files:
        local = os.path.join(folder, f['path'].replace('__', '/').split(key, 1)[1])
        assert sha(local) == f['sha256'], f'{name}: {local} differs from its frozen hash'
    print(name, len(files), 'files match their frozen hashes')

## Two queues, one per T4 (each resumes where it stopped)

In [ ]:
A = DETECTORS['channel-a']
SPANS = ['--gold-spans', GOLD_SPANS] if GOLD_SPANS else []
def run(name, *extra):
    return [sys.executable, '-m', 'faithguard.train.systems', '--items', ITEMS, '--out', f'{OUT}/systems-main-{name}', *extra]
queues = [
    [run('repair-v1', '--channel-a', A, '--adapter', ADAPTERS['repair-v1'], '--neural', NEURAL, *SPANS),
     run('repair-sft', '--channel-a', A, '--adapter', ADAPTERS['repair-sft']),
     run('channel-a-noslot', '--channel-a', DETECTORS['channel-a-noslot']),
     run('channel-a-noxbrl', '--channel-a', DETECTORS['channel-a-noxbrl'])],
    [run('repair-sft-nokeep', '--channel-a', A, '--adapter', ADAPTERS['repair-sft-nokeep']),
     run('stage-b', '--channel-a', A, '--adapter', ADAPTERS['stage-b']),
     run('rewrite-v1', '--channel-a', A, '--adapter', ADAPTERS['rewrite-v1'], '--mode', 'rewrite', *SPANS)],
]
script = 'import subprocess, sys, json\nfor cmd in json.loads(sys.argv[1]):\n    subprocess.run(cmd, check=True)\n'
import json
procs = [subprocess.Popen([sys.executable, '-c', script, json.dumps(q)], env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(i)},
                          stdout=open(f'{OUT}/queue-{i}.log', 'w'), stderr=subprocess.STDOUT) for i, q in enumerate(queues)]
codes = [p.wait() for p in procs]
print(codes)
for i in range(2):
    print(open(f'{OUT}/queue-{i}.log').read()[-3000:])
assert codes == [0, 0]

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])